Goal
- load a pretrained CNN
- understand its structure
- replace the classifier
- train only what’s necessary

### Choose a pretrained model (why ResNet-18)

We start with ResNet-18 because:

- small enough to run locally
- deep enough to be powerful
- widely used baseline
- easy to reason about

### Load the pretrained model

In [1]:
import torchvision.models as models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

`models.resnet18`
- constructs the architecture

`weights=ResNet18_Weights.DEFAULT`
- downloads pretrained ImageNet weights
- this replaces the old pretrained=True style

After this line
- the model already knows edges, textures, shapes

### Understand ResNet input expectations

ResNet expects

- input shape: `[batch, 3, 224, 224]`
- RGB images
- specific normalization

But CIFAR-10 images are

- `[3, 32, 32]`

So we must adapt the input, not the model

### Update transforms for pretrained models

Pretrained CNNs expect ImageNet normalization.

In [2]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize(224),
    # transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

`Resize(224)`
- upsamples CIFAR-10 images
- required by ResNet

`Normalize(mean, std)`
- matches ImageNet training statistics
- mandatory for pretrained weights

These correspond to:
- Red channel mean & std
- Green channel mean & std
- Blue channel mean & std

They were computed by:
- averaging millions of ImageNet images

These numbers describe the expected input distribution.

The numbers are fixed because:
- they are empirical statistics, not hyperparameters
- they describe the dataset the model was trained on
- changing them invalidates pretrained weights

Without this
- performance collapses


### Reload CIFAR-10 with new transforms

In [3]:
from torchvision import datasets

train_data = datasets.CIFAR10(
    root="data",
    train=True,
    transform=train_transform,
    download=False,
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    transform=test_transform,
    download=False,
)

### Replace the classifier

ResNet-18 ends with:

`Linear(in_features=512, out_features=1000)`

But CIFAR-10 has 10 classes.

So we replace it.

In [4]:
import torch.nn as nn

model.fc = nn.Linear(512, 10)

What this does

- keeps all convolutional layers
- removes ImageNet classifier
- adds a CIFAR-10 classifier

This is transfer learning.

### Freeze feature extractor

Freeze feature extractor

In [5]:
for param in model.parameters():
    param.requires_grad = False

Then allow only the classifier to train:

In [6]:
for param in model.fc.parameters():
    param.requires_grad = True

Why

- early layers already know vision
- training everything would be slow
- small dataset → overfitting risk

This makes training fast and stable.

### Loader

In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_data, 
    batch_size=128, 
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

### Move model to device

In [8]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
print(device)

cuda


### Loss and optimizer

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

Why only model.fc.parameters()

- only classifier is trainable
- optimizer should update only those weights

This avoids wasted computation.

### Training Loop

In [10]:
epochs = 5

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        loss = loss_fn(preds, labels)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        
    epoch_loss /= len(train_loader)
    
    print(f"Epoch: {epoch} | Loss per epoch: {epoch_loss:.4f}")

Epoch: 0 | Loss per epoch: 0.9012
Epoch: 1 | Loss per epoch: 0.6222
Epoch: 2 | Loss per epoch: 0.5863
Epoch: 3 | Loss per epoch: 0.5650
Epoch: 4 | Loss per epoch: 0.5549


### Testing Loop

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)
        predicted = preds.argmax(dim=1)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
    accuracy = correct / total
    print(accuracy)

10000
0.8076
